# Transform Circuits Data

1. Read bronze `circuits` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
1. Filter out rows where `circuit_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of columns `circuit_name` and `locality` to Title Case
1. Write the transformed data to silver `circuits` table

> Below changes are required to implement Incremental Load Processing
1. Accept batch_id as a parameter to the notebook
1. Process data for only the batch_id being passed in (i.e., filter reading from bronze using the batch_id)
1. Add created_timestamp, updated_timestamp and batch_id to the silver table. 
1. Merge the processed data to the silver table
    - created_timestamp should only be populated at the time of inserting/ creating the record. It should not be updated during the merge update.
    - Ensure that we are not overwriting the data in silver table by older bronze data (re-run scenario)

### Step 1 - Loading the env config from the common config

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.circuits"
silver_table=f"{catalog_name}.{silver_schema}.circuits"

In [0]:
from pyspark.sql import functions as f

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
circuits_df = spark.read.table(bronze_table).filter(f.col("batch_id") == v_batch_id)
display(circuits_df)

### Keep only the columns required for analytics (Drop url column)

In [0]:
circuits_selected_df=circuits_df.select(
    f.col("circuitId"),
    f.col("circuitName"),
    f.col("lat"),
    f.col("long"),
    f.col("locality"),
    f.col("country"),
    f.col("ingestion_timestamp"),
    f.col("source_file"),
    f.col("batch_id")
)

###Step 3 & 4 Standardise column names && rename columns to make them more meaningful
Standardise column names using snake_case ( circuitid → circuit_id, circuitNane → circuit_nane )
Rename columns to make them more meaningful ( lat → latitude, long → longitude)

In [0]:
# circuits_renamed_df=(
#     circuits_selected_df
#     .withColumnRenamed("circuitId","circuit_id")
#     .withColumnRenamed("circuitName","circuit_name")
#     .withColumnRenamed("lat","latitude")
#     .withColumnRenamed("long","longitude")
# )

In [0]:
circuits_renamed_df=(
    circuits_selected_df.withColumnsRenamed({
        "circuitId":"circuit_id",
        "circuitName":"circuit_name",
        "lat":"latitude",
        "long":"longitude"
        })
)

### Step 5 - Filter out rows where circult_1d is null (business key validation)

In [0]:
# circuits_valid_df=circuits_renamed_df.filter("circuit_id is not NULL")
# display(circuits_valid_df)


In [0]:
circuits_valid_df=circuits_renamed_df.filter(
    f.col("circuit_id").isNotNull()
)
display(circuits_valid_df)

### Step 6. Remove duplicate records

In [0]:
circuits_distinct_df=circuits_valid_df.distinct()
display(circuits_distinct_df)
# This removed the duplicate record from the table but we want to removed the duplicate cicuits_id

In [0]:
circuits_dup_df=circuits_valid_df.dropDuplicates(["circuit_id"])
display(circuits_dup_df)
# It will remove duplicate circuit_id value (but it is random)

### Step 7. Transform values of columns circuit_name and locality to Title Case

In [0]:
circuits_final_df=(
    circuits_dup_df
    .withColumn("circuit_name",f.initcap(f.col("circuit_name")))
    .withColumn("locality",f.initcap(f.col("locality")))
                )

In [0]:
write_to_silver(
    input_df=circuits_final_df,
    target_table=silver_table,
    merge_condition="t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
# circuits_final_df=(
#     circuits_final_df
#     .withColumn("created_timestamp",f.current_timestamp())
#     .withColumn("updated_timestamp",f.current_timestamp())
# )

In [0]:
# from delta.tables import DeltaTable

# if not spark.catalog.tableExists(silver_table):
#     circuits_final_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

# else:
#     delta_table = DeltaTable.forName(spark, silver_table)
# (
#     delta_table.alias("t")
#     .merge(circuits_final_df.alias("s"), "t.circuit_id=s.circuit_id")
#     .whenMatchedUpdate(
#         condition="s.batch_id >= t.batch_id",
#         set={
#             "circuit_name": "s.circuit_name",
#             "latitude": "s.latitude",
#             "longitude": "s.longitude",
#             "locality": "s.locality",
#             "country": "s.country",
#             "ingestion_timestamp": "s.ingestion_timestamp",
#             "source_file": "s.source_file",
#             "batch_id": "s.batch_id",
#             "updated_timestamp": "s.updated_timestamp",
#         },
#     )
#     .whenNotMatchedInsertAll()
#     .execute()
# )

In [0]:
spark.table(silver_table).show()